## Load modules

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import os
import pandas as pd
import lxml.etree as et


## Define paths and variable ranges

In [ ]:
# specify url and paths
page_url = 'https://archaeologydataservice.ac.uk/archsearch/advanced.xhtml'
driver_path = 'chromedriver'  # update this to your local chromedriver path, or leave as-is if chromedriver is on your PATH
download_path = '../output/ADS_search/downloads/'
xml_files_path = '../output/ADS_search/xml_files/'

# list of counties
counties = ['West Yorkshire', 'North Yorkshire', 'East Riding of Yorkshire', 'South Yorkshire', 
            'York', 'Northumberland', 'Tyne and Wear', 'Durham', 'Cumbria', 'Lancashire', 'Blackpool', 
            'Blackburn with Darwen', 'Darlington', 'Stockton-on-Tees', 'Middlesbrough', 'Hartlepool', 
            'Redcar and Cleveland', 'Kingston upon Hull', 'Greater Manchester', 'Merseyside', 'Halton', 
            'Warrington', 'Cheshire West and Chester', 'Cheshire East']

# define periods
periods = ['Roman', 'Iron Age']

# create log
results_log = []


## Run webdriver and get web page

In [ ]:
driver = webdriver.Chrome(executable_path=driver_path)
driver.implicitly_wait(10) # seconds wait
driver.get(page_url)


## Loop through periods and counties and download  results

In [ ]:
# first loop for periods
for period in periods:
    # enter temporal and county criteria and perform search
    temporal_input = driver.find_element_by_id('advancedSearchForm:temporal')
    temporal_input.clear()
    temporal_input.send_keys(period)
    
    # second loop for counties
    for county in counties:
        current_page = 1
        try:
            # create name prefix
            filename_prefix = county.lower().replace(' ', '_') + '_' + period.lower().replace(' ', '_')

            # enter county
            county_input = driver.find_element_by_id('advancedSearchForm:county')
            county_input.clear()
            county_input.send_keys(county)
            #time.wait(1)
            
            # clear keywords to be safe
            kw_input = driver.find_element_by_id('advancedSearchForm:keyword')
            kw_input.clear()
            
            # perform search
            county_input.send_keys(Keys.RETURN)
            
            # if it's the first county, expand "Object Types" chevron and select "Ecofacts"
            if county == counties[0]:
                ot_toggle = driver.find_element_by_xpath('//span[contains(text(), "Object Types")]/parent::*/parent::*/preceding-sibling::*/a')
                driver.execute_script("arguments[0].click();", ot_toggle)  # normal click won't work so we use javascript instead
            
                # select "Ecofacts"
                #ef_toggle = driver.find_element_by_xpath('//span[contains(text(), "Ecofacts")]/preceding-sibling::*/a')
                #driver.execute_script("arguments[0].click();", ef_toggle)  # normal click won't work so we use javascript instead
                ef_toggle = driver.find_element_by_xpath('//span[contains(text(), "Ecofacts")]/parent::*/parent::*/preceding-sibling::*/a')
                driver.execute_script("arguments[0].click();", ef_toggle)  # normal click won't work so we use javascript instead
                
                # select "Animal Remains"
                ar_toggle = driver.find_element_by_xpath('//span[contains(text(), "Animal Remains")]/preceding-sibling::*/a')
                driver.execute_script("arguments[0].click();", ar_toggle)  # normal click won't work so we use javascript instead
                
                
            if driver.find_elements_by_xpath('//*[@id="results"]'):
                # get number of result pages
                page_count = driver.find_element_by_xpath('//span[@class="resultsCount"]').text
                page_count = int(page_count.split()[-1])

                # loop through result pages and download xml file
                while current_page <= page_count:
                    # download data
                    driver.find_element_by_xpath('//a[@title="Download a page of results in XML"]').click()
                    old_name = download_path + 'result.xml'
                    new_name = xml_files_path + filename_prefix + '_' + str(current_page) + '.xml'

                    # move downloaded file
                    ren = 0
                    tries = 0
                    while ren == 0:
                        if os.path.exists(old_name):
                            os.rename(old_name, new_name)
                            ren = 1
                            continue
                        elif tries > 10:
                            driver.find_element_by_xpath('//a[@title="Download a page of results in XML"]').click()
                            tries = 0
                        else:
                            tries += 1
                            time.sleep(1)

                    # move to next page
                    current_page += 1
                    next_page = driver.find_element_by_xpath('//a[@title="Next page"]')
                    driver.execute_script("arguments[0].click();", next_page)

                # update log
                log_entry = county + ': ' + str(page_count) + ' pages processed succesfully'
                results_log.append(log_entry)

        except:
            # update log
            log_entry = county + ': processing error - page ' + str(current_page)
            results_log.append(log_entry)
        
        
        # load advanced search page
        driver.find_element_by_xpath('//*[@id="advancedBlade"]/a').click()

#close window
driver.close()

# write log to disk
with open('log.txt', 'w') as log_file:
    for entry in results_log:
        log_file.write(entry + '\n')

## Merge the xml files

In [ ]:
# merge xml files
files = [i.name for i in os.scandir(xml_files_path) if i.name != ".DS_Store"]

# remove older files
if os.path.exists('xslt.xsl'):
    os.remove('xslt.xsl')

if os.path.exists('full_results.xml'):
    os.remove('full_results.xml')

# create transformation sheet
with open('xslt.xsl', 'w') as xslt_file:
    xslt_file.write('<?xml version="1.0" ?>\n<xsl:transform xmlns:xsl="http://www.w3.org/1999/XSL/Transform" version="1.0">\n<xsl:template match="SearchResults">\n<xsl:copy>\n<xsl:copy-of select="Record"/>\n')

    for i, file in enumerate(files):
        if i != 0:
            copy_line = '<xsl:copy-of select="document(\'' + xml_files_path + file + '\')/SearchResults/Record"/>\n'
            xslt_file.write(copy_line)
        
    xslt_file.write('</xsl:copy>\n</xsl:template>\n</xsl:transform>')

# use sheet to merge xml files
with open('full_results.xml', 'wb') as results_file:
    first_file = xml_files_path + files[0]
    dom = et.parse(first_file)
    xslt = et.parse('xslt.xsl')
    transform = et.XSLT(xslt)
    newdom = transform(dom)
    tree_out = et.tostring(newdom, encoding='UTF-8', pretty_print=True,  xml_declaration=True)
    results_file.write(tree_out)


## Parse the XML and extract information into a dataframe

In [ ]:
# convert xml results to dataframe
xtree = et.parse('full_results.xml')
xroot = xtree.getroot()

# column headers for dataframe
header_list = ['UUID', 'Resource ID', 'Ref Count', 'References']
rows = []
uuid = 0
prefix_map = {"ns": "http://www.heritage-standards.org/midas/schema/1.0"}

for node in xroot:
    uuid += 1
    record_id = node.find('Meta/ResourceID').text
    ref_list = []
    if node.find('.//ns:references', prefix_map) is not None:
        for ref in node.find('.//ns:references', prefix_map):
            r = ref.find('.//ns:full', prefix_map)
            ref_list.append(r.text)
    #print(ref_list)
    if ref_list:
        rows.append({'UUID': uuid, 'Resource ID': record_id, 'Ref Count': len(ref_list), 'References': ', '.join(ref_list)})
    
df = pd.DataFrame(rows, columns=header_list)

# save results to CSV
df.to_csv('csv_results.csv', index=False)

# show dataframe
df